In [3]:
import os
import base64

from email import policy
from email.parser import BytesParser
from email.message import EmailMessage

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build


SCOPES = [
    "https://www.googleapis.com/auth/gmail.modify",
    "https://www.googleapis.com/auth/gmail.send",
]


def get_gmail_service():
    """
    Authenticate with Gmail and return the Gmail API service.
    """

    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file(
            "token.json",
            SCOPES
        )

    if not creds or not creds.valid:

        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())

        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json",
                SCOPES
            )

            creds = flow.run_local_server(port=0)

        with open("token.json", "w") as token:
            token.write(creds.to_json())

    return build("gmail", "v1", credentials=creds)

def extract_body(message):
    """
    Extract the readable text body from an email.
    """

    plain_text = []
    html_text = []

    if message.is_multipart():

        for part in message.walk():

            # Ignore attachments
            if part.get_content_disposition() == "attachment":
                continue

            content_type = part.get_content_type()

            try:
                content = part.get_content()
            except Exception:
                continue

            if content_type == "text/plain":
                plain_text.append(str(content))

            elif content_type == "text/html":
                html_text.append(str(content))

    else:

        try:
            content = message.get_content()
        except Exception:
            content = ""

        if message.get_content_type() == "text/plain":
            plain_text.append(str(content))

        elif message.get_content_type() == "text/html":
            html_text.append(str(content))

    if plain_text:
        return "\n".join(plain_text).strip()

    if html_text:
        return "\n".join(html_text).strip()

    return ""
def get_all_unread(service):
    """
    Get every unread email currently in the inbox.
    """

    messages = []
    page_token = None

    while True:

        result = (
            service.users()
            .messages()
            .list(
                userId="me",
                q="is:unread in:inbox",
                maxResults=500,
                pageToken=page_token,
            )
            .execute()
        )

        messages.extend(result.get("messages", []))

        page_token = result.get("nextPageToken")

        if not page_token:
            break

    return messages


from email.utils import parseaddr


def fetch_email(service, message_id):

    result = (
        service.users()
        .messages()
        .get(
            userId="me",
            id=message_id,
            format="raw"
        )
        .execute()
    )

    raw_data = base64.urlsafe_b64decode(
        result["raw"]
    )

    message = BytesParser(
        policy=policy.default
    ).parsebytes(raw_data)

    # Convert header objects into normal strings
    sender = str(message.get("From", ""))
    reply_to = str(message.get("Reply-To", ""))

    sender_email = parseaddr(sender)[1]
    reply_to_email = parseaddr(reply_to)[1]

    return {
        "id": message_id,
        "thread_id": result.get("threadId"),

        "from": sender,
        "from_email": sender_email,

        "reply_to": reply_to,
        "reply_to_email": reply_to_email,

        "to": str(message.get("To", "")),
        "subject": str(message.get("Subject", "")),
        "date": str(message.get("Date", "")),

        "message_id": str(
            message.get("Message-ID", "")
        ),

        "references": str(
            message.get("References", "")
        ),

        "body": extract_body(message),
    }

def mark_as_read(service, message_id):
    """
    Remove the UNREAD label from a Gmail message.
    """

    service.users().messages().modify(
        userId="me",
        id=message_id,
        body={
            "removeLabelIds": ["UNREAD"]
        }
    ).execute()


def send_email(
    service,
    to,
    subject,
    body,
    cc=None,
    bcc=None,
):
    """
    Send a new email.
    """

    message = EmailMessage()

    message["To"] = to
    message["Subject"] = subject

    if cc:
        message["Cc"] = cc

    if bcc:
        message["Bcc"] = bcc

    message.set_content(body)

    encoded_message = base64.urlsafe_b64encode(
        message.as_bytes()
    ).decode()

    result = (
        service.users()
        .messages()
        .send(
            userId="me",
            body={
                "raw": encoded_message
            }
        )
        .execute()
    )

    return result

def reply_to_email(
    service,
    original_email,
    
    subject=None,
    body=""
):
    """
    Reply to an existing Gmail email inside the same thread.

    original_email should be the dictionary returned by fetch_email().
    """

    # Prefer the Reply-To address if the sender provided one.
    # Otherwise reply to the normal From address.
    recipient = (
        original_email.get("reply_to_email")
        or original_email.get("from_email")
    )

    if not recipient:
        raise ValueError(
            "Could not determine who to reply to."
        )

    # Use the original subject unless you provide another one.
    original_subject = original_email.get(
        "subject",
        ""
    )

    if subject is None:
        if original_subject.lower().startswith("re:"):
            subject = original_subject
        else:
            subject = f"Re: {original_subject}"

    # Create the outgoing email.
    message = EmailMessage()

    message["To"] = recipient
    message["Subject"] = subject

    # These headers tell email clients that this is a reply
    # to the original message.
    original_message_id = original_email.get(
        "message_id",
        ""
    )

    original_references = original_email.get(
        "references",
        ""
    )

    if original_message_id:

        message["In-Reply-To"] = original_message_id

        references = (
            f"{original_references} "
            f"{original_message_id}"
        ).strip()

        message["References"] = references

    # Add the actual reply text.
    message.set_content(body)

    # Gmail API expects the email encoded as Base64.
    encoded_message = base64.urlsafe_b64encode(
        message.as_bytes()
    ).decode()

    request_body = {
        "raw": encoded_message
    }

    # This keeps the email inside the same Gmail conversation.
    thread_id = original_email.get(
        "thread_id"
    )

    if thread_id:
        request_body["threadId"] = thread_id

    # Send the reply.
    result = (
        service.users()
        .messages()
        .send(
            userId="me",
            body=request_body
        )
        .execute()
    )

    return result

def process_email(email_data):
    """
    Put your project's email processing logic here.

    Only return successfully if the email has actually
    been processed.
    """

    print()
    print("=" * 80)
    print("FROM:", email_data["from"])
    print("SUBJECT:", email_data["subject"])
    print("DATE:", email_data["date"])
    print()
    print(email_data["body"])

    # Your project logic goes here.


def main():

    service = get_gmail_service()

    unread_messages = get_all_unread(service)

    print(f"Found {len(unread_messages)} unread emails.")

    for item in unread_messages:

        message_id = item["id"]

        try:

            email_data = fetch_email(
                service,
                message_id
            )

            process_email(email_data)

            # Only mark as read AFTER successful processing.
            mark_as_read(
                service,
                message_id
            )

            print(
                f"Processed successfully: "
                f"{email_data['subject']}"
            )

        except Exception as error:

            print(
                f"Failed to process {message_id}: {error}"
            )


In [4]:
service = get_gmail_service()
allmessages = get_all_unread(service)

for x in allmessages:
    
    emaildata = fetch_email(service,x["id"])
   

    print(emaildata)

{'id': '1a09b192871b2c82', 'thread_id': '1a09b02aee329b50', 'from': 'AlsworthDoesntMiss <alsw.nidhin@gmail.com>', 'from_email': 'alsw.nidhin@gmail.com', 'reply_to': '', 'reply_to_email': '', 'to': 'als! <justtals.psd@gmail.com>', 'subject': 'Re: Request for Data Deletion Under UK Data Protection Laws', 'date': 'Sun, 13 Sep 2026 17:08:22 +0300', 'message_id': '<CAOqBGk18-cHf=DFRhyHoFfdr_=Ac=8-e7_4WE5Rd2GTZK7bkeA@mail.gmail.com>', 'references': '<CAN7QuZbrhcOUQEhhd0q0NmUkvn0FyM1YW-Qg4Op6-wvcE=B4Aw@mail.gmail.com> <CAOqBGk1batzH4S8RJpKaeiox7P00BFL8qrFRby06vZe9fTd7+Q@mail.gmail.com> <CAOqBGk0mB28txu04p2Yxy+BrY+au=KZrZ3LfRQJ+c2ZwQv51Nw@mail.gmail.com>', 'body': 'Dear akdjf,\r\n\r\nWe may or may not hold your data. We require further proof of\r\nidentification to prove your identity. We need your legal name, birth date,\r\nand postcode\r\nWe received our information from Osamason.\r\n\r\nRegards\r\n\r\nOn Sun, Sep 13, 2026 at 5:07\u202fPM AlsworthDoesntMiss <alsw.nidhin@gmail.com>\r\nwrote:\

In [44]:
print(emaildata)

{'id': '1a09b04bb9dc48ab', 'thread_id': '1a09b02aee329b50', 'from': 'AlsworthDoesntMiss <alsw.nidhin@gmail.com>', 'from_email': 'alsw.nidhin@gmail.com', 'reply_to': '', 'reply_to_email': '', 'to': 'als! <justtals.psd@gmail.com>', 'subject': 'Re: Request for Data Deletion Under UK Data Protection Laws', 'date': 'Sun, 13 Sep 2026 16:46:02 +0300', 'message_id': '<CAOqBGk1batzH4S8RJpKaeiox7P00BFL8qrFRby06vZe9fTd7+Q@mail.gmail.com>', 'references': '<CAN7QuZbrhcOUQEhhd0q0NmUkvn0FyM1YW-Qg4Op6-wvcE=B4Aw@mail.gmail.com>', 'body': 'Dear asldkfn,\r\n\r\nWe do hold your data, and received it from Haribo. We will delete your data\r\nwith immediate effect.\r\n\r\nRegards\r\n\r\n\r\nOn Sun, Sep 13, 2026 at 4:44\u202fPM als! <justtals.psd@gmail.com> wrote:\r\n\r\n> Dear Data Protection Officer,\r\n>\r\n> I am writing to inquire whether your organisation stores or processes\r\n> any personal data related to me. If you do hold any such information,\r\n> please inform me from which source(s) you obtained

In [7]:
 
prompt = "Here is the data from an email:" +str(emaildata) + ". Go through the email and classify it based on whether it is a " \
"personal email from another individual, a marketing email from a business/organisation, or if it is a reply to an email" \
" I previously sent. I am building a tool to identify data brokers who have my data and automate data deletion requests to each." \
" You must only output the classification as a one-word string i.e 'Personal', 'Marketing', or 'Reply'." 

In [16]:
print(prompt)

Here is the data from an email:{'id': '1a0981842e9e7d25', 'thread_id': '1a0981842e9e7d25', 'from': 'AlsworthDoesntMiss <alsw.nidhin@gmail.com>', 'from_email': 'alsw.nidhin@gmail.com', 'reply_to': '', 'reply_to_email': '', 'to': '"justtals.psd@gmail.com" <justtals.psd@gmail.com>', 'subject': 'gagain', 'date': 'Sun, 13 Sep 2026 03:08:30 +0300', 'message_id': '<CAOqBGk1is1v8mCWRnK+jUtOMYiACcuN9u1SBYgFnhznocZyq=w@mail.gmail.com>', 'references': '', 'body': 'yeat'}. Go through the email and classify it based on whether it is a personal email from another individual, a marketing email from a business/organisation, or if it is a reply to an email I previously sent. I am building a tool to identify data brokers who have my data and automate data deletion requests to each. You must only output the classification as a one-word string i.e 'Personal', 'Marketing', or 'Reply'.


In [1]:
import os

from openai import OpenAI


api_key = "dummy"
if not api_key:
    raise RuntimeError("Set the OPENAI_API_KEY environment variable before running this cell.")

client = OpenAI(api_key=api_key)

response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt)

print(response.output_text)

NameError: name 'prompt' is not defined

In [56]:
prompt2 = "Here is a Marketing email." + str(emaildata) + ". Output only the email address of the Data Protection Office of the organisation the" \
" email originated from, if one exists. If one does not exist, output the Customer Support or Contact email address. " \
". If none, output only the email address that the original email was sent from. Next, generate an appropriate subject for " \
"a data deletion request email to the organisation as per UK data protection laws. Then, generate the body of the email first" \
"asking whether the organisation stores any of my individual data, and if so, where they got the information from, and a deletion" \
"request." \
"email. All 3 fields must be output as a string containing only the requested information. These strings must then be" \
"stored in a dictionary with structure {'email_address': 'identified email address to send email to'" \
" 'subject': 'generated subject of the data deletion request email', 'body': 'generated body of the data deletion request email'}." 

In [57]:
#dicofbrokers = {}
if response.output_text == "Marketing":
    response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt2)


In [32]:
print(response.output_text)

{'email_address': 'alsw.nidhin@gmail.com', 'subject': 'Request for Data Deletion Under UK Data Protection Laws', 'body': 'Dear Data Protection Officer,\n\nI am writing to inquire whether your organisation stores or processes any personal data related to me. If you do hold any such information, please inform me from which source(s) you obtained my data.\n\nFurthermore, I would like to request the deletion of all personal data concerning me, as permitted under UK data protection laws.\n\nThank you for your assistance.\n\nBest regards,'}


In [58]:
import ast
data = ast.literal_eval(response.output_text)

send_email(service, data["email_address"], data["subject"], data["body"])  


ValueError: malformed node or string on line 1: <ast.Name object at 0x0000028972290DD0>

In [9]:
prompt3 = "Here is a reply email from a data broker." + str(emaildata) + "Identify whether the reply requires further details or processing on my end" \
"or if no further action is required. If a source for where the organisation received my data is specified, output the name of " \
"that organisation too. All outputs must be in the form of a string,i.e 'COMPLETE' or 'FURTHER ACTION' for the first output and simply" \
"the name of the organisation that acted as the source for the data. The output is to be stored in the form of a dictionary" \
"with structure {'status': 'COMPLETE' or 'FURTHER ACTION', 'datasource' : 'Name of source'}"  

In [10]:
orgdata = (emaildata)
if response.output_text == "Reply":
    response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt3)

print(response.output_text)

{'status': 'FURTHER ACTION', 'datasource': 'Osamason'}


In [14]:
from pathlib import Path
from openpyxl import Workbook, load_workbook

excel_path = Path("brokers_requiring_action.xlsx")

if excel_path.exists():
    workbook = load_workbook(excel_path)
    worksheet = workbook.active
else:
    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = "Brokers"
    worksheet.append(["Organisation", "Status"])

# Parse the model response
import ast
data = ast.literal_eval(response.output_text)

if data.get("status") == "FURTHER ACTION":
    organisation = (
       orgdata.get("reply_to_email") or
        orgdata.get("from_email")
        
    )

    worksheet.append([
        organisation,
        data["status"],
    ])

    workbook.save(excel_path)
    

    